In [11]:
import sys


sys.path.append('../')

from bunkatopics import Bunka
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import load_dataset
import random

# import umap
from umap.umap_ import UMAP # My personal Umap bugs so I use this one
from sentence_transformers import SentenceTransformer
random.seed(42)


model_name = "all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=model_name) # We recommend starting with a small model




In [12]:

#Scientific Litterature Data
dataset = load_dataset("CShorten/ML-ArXiv-Papers")["train"]["title"]
raw_docs = random.sample(dataset, 200)


projection_model = UMAP(
                n_components=5,
                random_state=42,
                n_neighbors=5,# I want to optimise the local structure (5 low, 25 high)
                min_dist = 0.3,
                metric = 'cosine') # I don't want to disperse embeddings

# #embedding_model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
# embedding_model = SentenceTransformer(model_name_or_path="Bunka/sentence_transformer_encoder")

projection_model.n_components

5

In [13]:
bunka = Bunka(embedding_model=embedding_model, 
                projection_model=projection_model)  # the language is automatically detected, make sure the embedding model is adapted

In [14]:
bunka.actual_dimensions

5

In [15]:
# Fit Bunka to your text data
bunka.fit(raw_docs)

2025-05-08 16:54:08 - Bunka - INFO - Processing 2916 tokens


2025-05-08 16:54:08 - Bunka - INFO - Detected language: English
2025-05-08 16:54:08 - Bunka - INFO - Embedding documents... (can take varying amounts of time depending on their size)
2025-05-08 16:54:08 - Bunka - INFO - Reducing dimensions to 5 using UMAP
2025-05-08 16:54:09 - Bunka - INFO - Creating separate 2D projection for visualization
2025-05-08 16:54:09 - Bunka - INFO - Extracting meaningful terms from documents...
100%|██████████| 200/200 [00:00<00:00, 225.90it/s]


In [16]:
bunka.fig_embeddings

In [17]:
len(bunka.docs[0].embedding)
len(bunka.docs[0].nd_embedding)

5

In [18]:
from sklearn.cluster import HDBSCAN

max_cluster_size = int(0.02*len(raw_docs))
min_cluster_size = max(2, int(0.003*len(raw_docs)))
# min_cluster_size = 15

clustering_model = HDBSCAN(min_samples = 1, 
                max_cluster_size=max_cluster_size, 
                min_cluster_size=min_cluster_size, 
                metric = 'euclidean',
                cluster_selection_method = 'leaf')


df_topics = bunka.get_topics(n_clusters=10, name_length=5, min_count_terms = 2,  custom_clustering_model=clustering_model) # Specify the number of terms to describe each topic

2025-05-08 16:54:11 - Bunka - INFO - There is not enough data to select terms with a minimum occurrence of 2. Setting min_count_terms to 1
2025-05-08 16:54:11 - Bunka - INFO - Computing the topics
2025-05-08 16:54:11 - Bunka - INFO - Using 5-dimensional topic modeling
2025-05-08 16:54:11 - Bunka - INFO - Performing topic modeling using 5-dimensional embeddings
2025-05-08 16:54:11 - Bunka - INFO - Clustering 200 documents in 5D space
2025-05-08 16:54:11 - Bunka - INFO - Reprojecting n-dimensional embeddings to 2D for visualization
2025-05-08 16:54:11 - Bunka - INFO - Successfully reprojected document embeddings to 2D
2025-05-08 16:54:11 - Bunka - INFO - Successfully calculated topic 2D centroids from document positions


In [19]:
bunka.visualize_topics()

2025-05-08 16:54:11 - Bunka - INFO - Creating the Bunka Map


In [21]:
## Add it if it is 2 emebnddings

bunka.docs[0].nd_embedding

[4.53045129776001,
 6.187203884124756,
 6.059620380401611,
 7.609132766723633,
 6.218442916870117]